In [1]:
#Shishir Khanal
#Jul 11, 2026
#Black Jack RL Agent

In [4]:
import numpy as np
import gym

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [13]:
class Agent():
    def __init__(self, gamma=0.99):
        self.V = {} # Keep track of agent's estimates of values of states
        self.sum_space = [i for i in range(4, 22)] # Possible sums of player's cards
        self.dealer_show_card_space = [i+1 for i in range(10)] #Possible cards in dealer's hand
        self.action_space = [False, True] # Stick/hit
        self.ace_space = [False, True] #Whether player has a usable ace
        
        self.state_space = [] # Track first visit to each state
        self.returns = {} # Track discounted returns after visit to each state
        self.states_visited = {} #first visit or not
        self.memory = [] # states encountered and awards received
        self.gamma = gamma # discount factor

        self.init_vals()

    def init_vals(self):
        for total in self.sum_space:
            for card in self.dealer_show_card_space:
                for ace in self.ace_space:
                    self.V[(total, card, ace)] = 0
                    self.returns[(total, card, ace)] = [] # returns over each of states over course of all episodes
                    self.states_visited[(total, card, ace)] = 0
                    self.state_space.append((total, card, ace))

    def policy(self, state):
        total, _, _ = state
        action = 0 if total >=20 else 1
        return action

    def update_V(self):
        for idt, (state,_) in enumerate(self.memory):
            G = 0
            if self.states_visited[state] == 0:
                self.states_visited[state] += 1
                discount = 1
                for t, (_, reward) in enumerate(self.memory[idt:]):
                    G += reward * discount
                    discount *= self.gamma
                    self.returns[state].append(G)

        for state, _ in self.memory:
            self.V[state] = np.mean(self.returns[state])

        for state in self.state_space:
            self.states_visited[state] = 0

        self.memory = []

In [14]:
if __name__ == '__main__':
    env = gym.make('Blackjack-v1')
    agent = Agent()
    n_episodes = 500000
    for i in range(n_episodes):
        if i % 50000 == 0:
            print('starting_episode', i)
        observation, info = env.reset()
        done = False
        while not done:
            action = agent.policy(observation)
            observation_, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            agent.memory.append((observation, reward))
            observation = observation_
        agent.update_V()

    print(agent.V[(21,3,True)])
    print(agent.V[(4, 1, False)])

starting_episode 0
starting_episode 50000
starting_episode 100000
starting_episode 150000
starting_episode 200000
starting_episode 250000
starting_episode 300000
starting_episode 350000
starting_episode 400000
starting_episode 450000
0.9736506587335316
-0.1793113195020747
